In [1]:
from pathlib import Path
import sys

# Add notebook directory so reactor_parameters can be found
_cwd = Path.cwd().resolve()
_candidates = [_cwd, *_cwd.parents]
_nb_dir = next((p for p in _candidates if (p / "reactor_parameters.py").is_file()), None)
if _nb_dir is not None and str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

import numpy as np
import matplotlib.pyplot as plt

from reactor_parameters import *


# reproduce T-seeded startup case (no He3,He4)

In [2]:
species_params_dd = {
    "D": {
        "f_0": 1.0,
        "tau_p": tau_p_T,
        "tau_ifc": np.inf,
        "tau_ofc": np.inf,
        "N_stor_min": 0.0,
        "Ndot_max": np.inf,
        "N_ofc_0": float(SPECIES_DEFAULTS["D"]["N_ofc_0"]),
        "N_ifc_0": float(SPECIES_DEFAULTS["D"]["N_ifc_0"]),
        "N_stor_0": float(SPECIES_DEFAULTS["D"]["N_stor_0"]),
        "enable_plasma_channel": True,
    },
    "T": {
        "f_0": 0.0,
        "tau_p": tau_p_T,
        "tau_ifc": 4.0 * 3600.0,
        "tau_ofc": 7200.0,
        "N_stor_min": 0.001 / species_mass["T"],
        "Ndot_max": Ndot_max_T,
        "N_ofc_0": float(SPECIES_DEFAULTS["T"]["N_ofc_0"]),
        "N_ifc_0": float(SPECIES_DEFAULTS["T"]["N_ifc_0"]),
        "N_stor_0": float(SPECIES_DEFAULTS["T"]["N_stor_0"]),
        "enable_plasma_channel": True,
    },
    "He3": {
        "f_0": 0.0,
        "tau_p": tau_p_T,
        "tau_ifc": np.inf,
        "tau_ofc": np.inf,
        "N_stor_min": 0.0,
        "Ndot_max": 0.0,
        "N_ofc_0": float(SPECIES_DEFAULTS["He3"]["N_ofc_0"]),
        "N_ifc_0": float(SPECIES_DEFAULTS["He3"]["N_ifc_0"]),
        "N_stor_0": float(SPECIES_DEFAULTS["He3"]["N_stor_0"]),
        "enable_plasma_channel": False,
    },
    "He4": {
        "f_0": 0.0,
        "tau_p": tau_p_T,
        "tau_ifc": np.inf,
        "tau_ofc": np.inf,
        "N_stor_min": 0.0,
        "Ndot_max": 0.0,
        "N_ofc_0": float(SPECIES_DEFAULTS["He4"]["N_ofc_0"]),
        "N_ifc_0": float(SPECIES_DEFAULTS["He4"]["N_ifc_0"]),
        "N_stor_0": float(SPECIES_DEFAULTS["He4"]["N_stor_0"]),
        "enable_plasma_channel": False,
    },
}

inject_from_storage_dd = {"D": False, "T": True, "He3": False, "He4": False}
injection_mode_dd = {"D": "auto", "T": "off", "He3": "off", "He4": "off"}

species_params = {
    sp: {
        "tau_p": float(species_params_dd[sp]["tau_p"]),
        "lambda_decay": float(species_params_dd[sp].get("lambda_decay", SPECIES_DEFAULTS[sp]["lambda_decay"])),
        "tau_ifc": float(species_params_dd[sp]["tau_ifc"]),
        "tau_ofc": float(species_params_dd[sp]["tau_ofc"]),
        "N_stor_min": float(species_params_dd[sp]["N_stor_min"]),
        "Ndot_max": float(species_params_dd[sp]["Ndot_max"]),
        "inject_from_storage": bool(species_params_dd[sp].get("inject_from_storage", inject_from_storage_dd[sp])),
        "injection_mode": str(species_params_dd[sp].get("injection_mode", injection_mode_dd[sp])),
        "enable_plasma_channel": bool(species_params_dd[sp]["enable_plasma_channel"]),
    }
    for sp in SPECIES
}

initial_conditions = {
    sp: {
        "f_0": float(species_params_dd[sp]["f_0"]),
        "N_ofc_0": float(species_params_dd[sp]["N_ofc_0"]),
        "N_ifc_0": float(species_params_dd[sp]["N_ifc_0"]),
        "N_stor_0": float(species_params_dd[sp]["N_stor_0"]),
    }
    for sp in SPECIES
}

In [ ]:
# Solver dictionaries are defined above.

In [ ]:

reactivities_dd = compute_reactivities_from_functions(float(T_i))



In [4]:
# ----- Solve -----
res_dd = solve_multispecies_ode_system(
    V_plasma=float(V_plasma),
    T_i=float(T_i),
    n_tot=float(n_tot),
    species_params=species_params,
    initial_conditions=initial_conditions,
    TBR_DT=TBR_DT,
    TBR_DDn=TBR_DD,
    max_simulation_time=float(max_simulation_time),
    vector_length=int(vector_length),
    target_conditions=[{"target_specie": "T", "metric": "fraction", "value": 0.5}],
    reactivities=reactivities_dd,
)

t_startup_dd = float(res_dd["t_startup"])
if not np.isfinite(t_startup_dd):
    t_startup_dd = float(np.asarray(res_dd["t"])[-1]) if len(np.asarray(res_dd["t"])) else np.nan



In [5]:
# Compute fusion power profiles
_n_D = np.asarray(res_dd["n_D"], dtype=float)
_n_T = np.asarray(res_dd["n_T"], dtype=float)
_n_He3 = np.asarray(res_dd["n_He3"], dtype=float)
_E = REACTION_ENERGY_BY_CHANNEL
P_DDn = 0.5 * _n_D**2 * reactivities_dd["sigmav_DD_n"] * V_plasma * _E["sigmav_DD_n"]
P_DDp = 0.5 * _n_D**2 * reactivities_dd["sigmav_DD_p"] * V_plasma * _E["sigmav_DD_p"]
P_DT = _n_D * _n_T * reactivities_dd["sigmav_DT"] * V_plasma * _E["sigmav_DT"]
P_DHe3 = _n_D * _n_He3 * reactivities_dd["sigmav_DHe3"] * V_plasma * _E["sigmav_DHe3"]
P_TT = 0.5 * _n_T**2 * reactivities_dd["sigmav_TT"] * V_plasma * _E["sigmav_TT"]
P_He3He3 = 0.5 * _n_He3**2 * reactivities_dd["sigmav_He3He3"] * V_plasma * _E["sigmav_He3He3"]
P_THe3_ch1 = _n_T * _n_He3 * reactivities_dd["sigmav_THe3_ch1"] * V_plasma * _E["sigmav_THe3_ch1"]
P_THe3_ch2 = _n_T * _n_He3 * reactivities_dd["sigmav_THe3_ch2"] * V_plasma * _E["sigmav_THe3_ch2"]
P_THe3_ch3 = _n_T * _n_He3 * reactivities_dd["sigmav_THe3_ch3"] * V_plasma * _E["sigmav_THe3_ch3"]
P_fusion_total = P_DDn + P_DDp + P_DT + P_DHe3 + P_TT + P_He3He3 + P_THe3_ch1 + P_THe3_ch2 + P_THe3_ch3



In [6]:
# Original ddstartup T-seeded reference solver
orig = solve_ode_system(
    V_plasma=V_plasma,
    n_tot=n_tot,
    tau_p_T=species_params_dd["T"]["tau_p"],
    TBR_DT=TBR_DT,
    TBR_DDn=TBR_DD,
    tau_ifc=species_params_dd["T"]["tau_ifc"],
    tau_ofc=species_params_dd["T"]["tau_ofc"],
    sigmav_DD_p=sigmav_DD_p_ref,
    sigmav_DD_n=sigmav_DD_n_ref,
    sigmav_DT=sigmav_DT_ref,
    injection_rate_max=Ndot_max_T,
    max_simulation_time=max_simulation_time,
    N_stor_min=species_params_dd["T"]["N_stor_min"],
)

print("DDstartup reproduction check")
print(f"  original success = {orig['sol_success']}")
print(f"  multispecies success = {res_dd['sol_success']}")
print(f"  original t_startup = {orig['t_startup']:.6e} s ({orig['t_startup']/86400.0:.3f} d)")
print(f"  multispecies t_startup = {res_dd['t_startup']:.6e} s ({res_dd['t_startup']/86400.0:.3f} d)")
if np.isfinite(orig["t_startup"]) and np.isfinite(res_dd["t_startup"]):
    rel = (res_dd["t_startup"] - orig["t_startup"]) / orig["t_startup"]
    print(f"  relative difference = {100.0 * rel:.2f}%")

print(f"  max |sum(dn/dt)| = {np.nanmax(np.abs(res_dd.get('sum_dn_dt', np.array([np.nan])))):.6e} m^-3 s^-1")
print(f"  max |normalized residual| = {np.nanmax(np.abs(res_dd.get('normalized_residual', np.array([np.nan])))):.6e}")
print(f"  max |n_tot drift| = {np.nanmax(np.abs(res_dd.get('n_total_rel_drift', np.array([np.nan])))):.6e}")



DDstartup reproduction check
  original success = True
  multispecies success = False
  original t_startup = 8.465478e+06 s (97.980 d)
  multispecies t_startup = inf s (inf d)
  max |sum(dn/dt)| = nan m^-3 s^-1
  max |normalized residual| = nan
  max |n_tot drift| = nan


/tmp/ipykernel_89981/443680725.py:27: RuntimeWarning: All-NaN slice encountered
  print(f"  max |sum(dn/dt)| = {np.nanmax(np.abs(res_dd.get('sum_dn_dt', np.array([np.nan])))):.6e} m^-3 s^-1")
/tmp/ipykernel_89981/443680725.py:28: RuntimeWarning: All-NaN slice encountered
  print(f"  max |normalized residual| = {np.nanmax(np.abs(res_dd.get('normalized_residual', np.array([np.nan])))):.6e}")
/tmp/ipykernel_89981/443680725.py:29: RuntimeWarning: All-NaN slice encountered
  print(f"  max |n_tot drift| = {np.nanmax(np.abs(res_dd.get('n_total_rel_drift', np.array([np.nan])))):.6e}")


In [ ]:
t_ref = np.asarray(res_dd["t"], dtype=float)
n_D = np.maximum(np.asarray(res_dd.get("n_D", np.zeros_like(t_ref)), dtype=float), 0.0)
n_T = np.maximum(np.asarray(res_dd.get("n_T", np.zeros_like(t_ref)), dtype=float), 0.0)
n_He3 = np.maximum(np.asarray(res_dd.get("n_He3", np.zeros_like(t_ref)), dtype=float), 0.0)

(
    P_DDn,
    P_DDp,
    P_DT,
    P_DHe3,
    P_TT,
    P_He3He3,
    P_THe3_ch1,
    P_THe3_ch2,
    P_THe3_ch3,
    P_DT_eq,
) = compute_fusion_power_profiles_numba(
    n_D,
    n_T,
    n_He3,
    float(n_tot),
    float(V_plasma),
    float(reactivities_dd["sigmav_DD_p"]),
    float(reactivities_dd["sigmav_DD_n"]),
    float(reactivities_dd["sigmav_DT"]),
    float(reactivities_dd["sigmav_DHe3"]),
    float(reactivities_dd["sigmav_TT"]),
    float(reactivities_dd["sigmav_He3He3"]),
    float(reactivities_dd["sigmav_THe3_ch1"]),
    float(reactivities_dd["sigmav_THe3_ch2"]),
    float(reactivities_dd["sigmav_THe3_ch3"]),
)

P_fusion_total = P_DDn + P_DDp + P_DT + P_DHe3 + P_TT + P_He3He3 + P_THe3_ch1 + P_THe3_ch2 + P_THe3_ch3
P_DDn = np.asarray(P_DDn, dtype=float)
P_DDp = np.asarray(P_DDp, dtype=float)
P_DT = np.asarray(P_DT, dtype=float)
P_DHe3 = np.asarray(P_DHe3, dtype=float)
P_TT = np.asarray(P_TT, dtype=float)
P_He3He3 = np.asarray(P_He3He3, dtype=float)
P_THe3_ch1 = np.asarray(P_THe3_ch1, dtype=float)
P_THe3_ch2 = np.asarray(P_THe3_ch2, dtype=float)
P_THe3_ch3 = np.asarray(P_THe3_ch3, dtype=float)
P_DT_eq = np.asarray(P_DT_eq, dtype=float)
P_fusion_total = np.asarray(P_fusion_total, dtype=float)


In [ ]:
# Plasma-fraction comparison
fig, ax = plt.subplots(1, 1, figsize=(9, 4.5))
t_days = np.asarray(res_dd["t"], dtype=float) / 86400.0
for sp in SPECIES:
    key = f"n_{sp}"
    if key in res_dd:
        ax.plot(t_days, np.asarray(res_dd[key], dtype=float) / n_tot, label=f"f_{sp}")
total = np.zeros_like(t_days, dtype=float)
for sp in SPECIES:
    key = f"n_{sp}"
    if key in res_dd:
        total = total + np.asarray(res_dd[key], dtype=float)
ax.plot(t_days, total / n_tot, lw=2.2, ls='-.', label="f_total")
ax.plot(orig["t"] / 86400.0, orig["n_T"] / n_tot, "k--", lw=1.8, label="f_T original ref")
ax.axhline(0.5, color="gray", linestyle=":", label="target=0.5")
ax.set_xlabel("Time (days)")
ax.set_ylabel("Plasma fraction (-)")
ax.set_ylim(bottom=0.0)
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Total inventory comparison
fig, ax = plt.subplots(1, 1, figsize=(9, 4.5))
t_days = np.asarray(res_dd["t"], dtype=float) / 86400.0
for sp in SPECIES:
    total_inventory = None
    for comp in ["ofc", "ifc", "stor"]:
        key = f"N_{comp}_{sp}"
        if key in res_dd:
            arr = np.asarray(res_dd[key], dtype=float)
            total_inventory = arr if total_inventory is None else total_inventory + arr
    if total_inventory is not None:
        ax.plot(t_days, total_inventory * species_mass[sp], label=f"I_OFC+IFC+STOR_{sp}")
I_T_orig = (orig["N_ofc"] + orig["N_ifc"] + orig["N_stor"]) * species_mass["T"]
ax.plot(orig["t"] / 86400.0, I_T_orig, "k--", lw=1.8, label="I_T original ref")
ax.set_xlabel("Time (days)")
ax.set_ylabel("Total inventory (kg)")
ax.set_yscale("log")
ax.set_ylim(bottom=1e-6)
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Fusion powers
fig, ax = plt.subplots(1, 1, figsize=(9, 4.5))
t_days = np.asarray(res_dd["t"], dtype=float) / 86400.0
reaction_series = [
    ("P_DDn", P_DDn),
    ("P_DDp", P_DDp),
    ("P_DT", P_DT),
    ("P_DHe3", P_DHe3),
    ("P_TT", P_TT),
    ("P_He3He3", P_He3He3),
    ("P_THe3_ch1", P_THe3_ch1),
    ("P_THe3_ch2", P_THe3_ch2),
    ("P_THe3_ch3", P_THe3_ch3),
    ("P_fusion_total", P_fusion_total),
]
for label, series in reaction_series:
    ax.plot(t_days, np.asarray(series, dtype=float) * 1.0e-6, label=label)
ax.set_xlabel("Time (days)")
ax.set_ylabel("Power (MW)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


# Reproduce T-seeded startup case WITH He3, He4

In [8]:
species_params_dd = {
    "D": {
        "f_0": 1.0,
        "tau_p": tau_p_T,
        "tau_ifc": 4.0 * 3600.0,
        "tau_ofc": np.inf,
        "N_stor_min": 0.0,
        "Ndot_max": np.inf,
        "N_ofc_0": float(SPECIES_DEFAULTS["D"]["N_ofc_0"]),
        "N_ifc_0": float(SPECIES_DEFAULTS["D"]["N_ifc_0"]),
        "N_stor_0": float(SPECIES_DEFAULTS["D"]["N_stor_0"]),
        "enable_plasma_channel": True,
    },
    "T": {
        "f_0": 0.0,
        "tau_p": tau_p_T,
        "tau_ifc": 4.0 * 3600.0,
        "tau_ofc": 7200.0,
        "N_stor_min": 0.001 / species_mass["T"],
        "Ndot_max": Ndot_max_T,
        "N_ofc_0": float(SPECIES_DEFAULTS["T"]["N_ofc_0"]),
        "N_ifc_0": float(SPECIES_DEFAULTS["T"]["N_ifc_0"]),
        "N_stor_0": float(SPECIES_DEFAULTS["T"]["N_stor_0"]),
        "enable_plasma_channel": True,
    },
    "He3": {
        "f_0": 0.0,
        "tau_p": tau_p_T,
        "tau_ifc": 4.0 * 3600.0,
        "tau_ofc": np.inf,
        "N_stor_min": 0.0,
        "Ndot_max": 0.0,
        "N_ofc_0": float(SPECIES_DEFAULTS["He3"]["N_ofc_0"]),
        "N_ifc_0": float(SPECIES_DEFAULTS["He3"]["N_ifc_0"]),
        "N_stor_0": float(SPECIES_DEFAULTS["He3"]["N_stor_0"]),
        "enable_plasma_channel": True,
    },
    "He4": {
        "f_0": 0.0,
        "tau_p": tau_p_T,
        "tau_ifc": 4.0 * 3600.0,
        "tau_ofc": np.inf,
        "N_stor_min": 0.0,
        "Ndot_max": 0.0,
        "N_ofc_0": float(SPECIES_DEFAULTS["He4"]["N_ofc_0"]),
        "N_ifc_0": float(SPECIES_DEFAULTS["He4"]["N_ifc_0"]),
        "N_stor_0": float(SPECIES_DEFAULTS["He4"]["N_stor_0"]),
        "enable_plasma_channel": True,
    },
}

inject_from_storage_dd = {"D": False, "T": True, "He3": False, "He4": False}
injection_mode_dd = {"D": "auto", "T": "off", "He3": "off", "He4": "off"}

species_params = {
    sp: {
        "tau_p": float(species_params_dd[sp]["tau_p"]),
        "lambda_decay": float(species_params_dd[sp].get("lambda_decay", SPECIES_DEFAULTS[sp]["lambda_decay"])),
        "tau_ifc": float(species_params_dd[sp]["tau_ifc"]),
        "tau_ofc": float(species_params_dd[sp]["tau_ofc"]),
        "N_stor_min": float(species_params_dd[sp]["N_stor_min"]),
        "Ndot_max": float(species_params_dd[sp]["Ndot_max"]),
        "inject_from_storage": bool(species_params_dd[sp].get("inject_from_storage", inject_from_storage_dd[sp])),
        "injection_mode": str(species_params_dd[sp].get("injection_mode", injection_mode_dd[sp])),
        "enable_plasma_channel": bool(species_params_dd[sp]["enable_plasma_channel"]),
    }
    for sp in SPECIES
}

initial_conditions = {
    sp: {
        "f_0": float(species_params_dd[sp]["f_0"]),
        "N_ofc_0": float(species_params_dd[sp]["N_ofc_0"]),
        "N_ifc_0": float(species_params_dd[sp]["N_ifc_0"]),
        "N_stor_0": float(species_params_dd[sp]["N_stor_0"]),
    }
    for sp in SPECIES
}

In [ ]:
# Solver dictionaries are defined above.

In [ ]:

reactivities_dd = compute_reactivities_from_functions(float(T_i))



In [10]:
# ----- Solve -----
res_dd = solve_multispecies_ode_system(
    V_plasma=float(V_plasma),
    T_i=float(T_i),
    n_tot=float(n_tot),
    species_params=species_params,
    initial_conditions=initial_conditions,
    TBR_DT=TBR_DT,
    TBR_DDn=TBR_DD,
    max_simulation_time=float(max_simulation_time),
    vector_length=int(vector_length),
    target_conditions=[{"target_specie": "T", "metric": "fraction", "value": 0.5}],
    reactivities=reactivities_dd,
)

t_startup_dd = float(res_dd["t_startup"])
if not np.isfinite(t_startup_dd):
    t_startup_dd = float(np.asarray(res_dd["t"])[-1]) if len(np.asarray(res_dd["t"])) else np.nan

# Compute fusion power profiles using shared helper


In [11]:
# ----- Compute power -----
t_ref = np.asarray(res_dd["t"], dtype=float)
n_D = np.maximum(np.asarray(res_dd.get("n_D", np.zeros_like(t_ref)), dtype=float), 0.0)
n_T = np.maximum(np.asarray(res_dd.get("n_T", np.zeros_like(t_ref)), dtype=float), 0.0)
n_He3 = np.maximum(np.asarray(res_dd.get("n_He3", np.zeros_like(t_ref)), dtype=float), 0.0)

(
    P_DDn,
    P_DDp,
    P_DT,
    P_DHe3,
    P_TT,
    P_He3He3,
    P_THe3_ch1,
    P_THe3_ch2,
    P_THe3_ch3,
    P_DT_eq,
) = compute_fusion_power_profiles_numba(
    n_D,
    n_T,
    n_He3,
    float(n_tot),
    float(V_plasma),
    float(reactivities_dd["sigmav_DD_p"]),
    float(reactivities_dd["sigmav_DD_n"]),
    float(reactivities_dd["sigmav_DT"]),
    float(reactivities_dd["sigmav_DHe3"]),
    float(reactivities_dd["sigmav_TT"]),
    float(reactivities_dd["sigmav_He3He3"]),
    float(reactivities_dd["sigmav_THe3_ch1"]),
    float(reactivities_dd["sigmav_THe3_ch2"]),
    float(reactivities_dd["sigmav_THe3_ch3"]),
)

P_fusion_total = P_DDn + P_DDp + P_DT + P_DHe3 + P_TT + P_He3He3 + P_THe3_ch1 + P_THe3_ch2 + P_THe3_ch3
P_DDn = np.asarray(P_DDn, dtype=float)
P_DDp = np.asarray(P_DDp, dtype=float)
P_DT = np.asarray(P_DT, dtype=float)
P_DHe3 = np.asarray(P_DHe3, dtype=float)
P_TT = np.asarray(P_TT, dtype=float)
P_He3He3 = np.asarray(P_He3He3, dtype=float)
P_THe3_ch1 = np.asarray(P_THe3_ch1, dtype=float)
P_THe3_ch2 = np.asarray(P_THe3_ch2, dtype=float)
P_THe3_ch3 = np.asarray(P_THe3_ch3, dtype=float)
P_DT_eq = np.asarray(P_DT_eq, dtype=float)
P_fusion_total = np.asarray(P_fusion_total, dtype=float)

# Original ddstartup T-seeded reference solver
orig = solve_ode_system(
    V_plasma=V_plasma,
    n_tot=n_tot,
    tau_p_T=species_params_dd["T"]["tau_p"],
    TBR_DT=TBR_DT,
    TBR_DDn=TBR_DD,
    tau_ifc=species_params_dd["T"]["tau_ifc"],
    tau_ofc=species_params_dd["T"]["tau_ofc"],
    sigmav_DD_p=sigmav_DD_p_ref,
    sigmav_DD_n=sigmav_DD_n_ref,
    sigmav_DT=sigmav_DT_ref,
    injection_rate_max=Ndot_max_T,
    max_simulation_time=max_simulation_time,
    N_stor_min=species_params_dd["T"]["N_stor_min"],
)

print("DDstartup reproduction check (WITH He3, He4)")
print(f"  original success = {orig['sol_success']}")
print(f"  multispecies success = {res_dd['sol_success']}")
print(f"  original t_startup = {orig['t_startup']:.6e} s ({orig['t_startup']/86400.0:.3f} d)")
print(f"  multispecies t_startup = {res_dd['t_startup']:.6e} s ({res_dd['t_startup']/86400.0:.3f} d)")
if np.isfinite(orig["t_startup"]) and np.isfinite(res_dd["t_startup"]):
    rel = (res_dd["t_startup"] - orig["t_startup"]) / orig["t_startup"]
    print(f"  relative difference = {100.0 * rel:.2f}%")

print(f"  max |sum(dn/dt)| = {np.nanmax(np.abs(res_dd.get('sum_dn_dt', np.array([np.nan])))):.6e} m^-3 s^-1")
print(f"  max |normalized residual| = {np.nanmax(np.abs(res_dd.get('normalized_residual', np.array([np.nan])))):.6e}")
print(f"  max |n_tot drift| = {np.nanmax(np.abs(res_dd.get('n_total_rel_drift', np.array([np.nan])))):.6e}")



DDstartup reproduction check (WITH He3, He4)
  original success = True
  multispecies success = True
  original t_startup = 8.465478e+06 s (97.980 d)
  multispecies t_startup = 8.525734e+06 s (98.677 d)
  relative difference = 0.71%
  max |sum(dn/dt)| = nan m^-3 s^-1
  max |normalized residual| = nan
  max |n_tot drift| = nan


/tmp/ipykernel_89981/995704412.py:35: RuntimeWarning: All-NaN slice encountered
  print(f"  max |sum(dn/dt)| = {np.nanmax(np.abs(res_dd.get('sum_dn_dt', np.array([np.nan])))):.6e} m^-3 s^-1")
/tmp/ipykernel_89981/995704412.py:36: RuntimeWarning: All-NaN slice encountered
  print(f"  max |normalized residual| = {np.nanmax(np.abs(res_dd.get('normalized_residual', np.array([np.nan])))):.6e}")
/tmp/ipykernel_89981/995704412.py:37: RuntimeWarning: All-NaN slice encountered
  print(f"  max |n_tot drift| = {np.nanmax(np.abs(res_dd.get('n_total_rel_drift', np.array([np.nan])))):.6e}")


In [ ]:
# Plasma-fraction comparison
fig, ax = plt.subplots(1, 1, figsize=(9, 4.5))
t_days = np.asarray(res_dd["t"], dtype=float) / 86400.0
for sp in SPECIES:
    key = f"n_{sp}"
    if key in res_dd:
        ax.plot(t_days, np.asarray(res_dd[key], dtype=float) / n_tot, label=f"f_{sp}")
total = np.zeros_like(t_days, dtype=float)
for sp in SPECIES:
    key = f"n_{sp}"
    if key in res_dd:
        total = total + np.asarray(res_dd[key], dtype=float)
ax.plot(t_days, total / n_tot, lw=2.2, ls='-.', label="f_total")
ax.plot(orig["t"] / 86400.0, orig["n_T"] / n_tot, "k--", lw=1.8, label="f_T original ref")
ax.axhline(0.5, color="gray", linestyle=":", label="target=0.5")
ax.set_xlabel("Time (days)")
ax.set_ylabel("Plasma fraction (-)")
ax.set_ylim(bottom=0.0)
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Total inventory comparison
fig, ax = plt.subplots(1, 1, figsize=(9, 4.5))
t_days = np.asarray(res_dd["t"], dtype=float) / 86400.0
for sp in SPECIES:
    total_inventory = None
    for comp in ["ofc", "ifc", "stor"]:
        key = f"N_{comp}_{sp}"
        if key in res_dd:
            arr = np.asarray(res_dd[key], dtype=float)
            total_inventory = arr if total_inventory is None else total_inventory + arr
    if total_inventory is not None:
        ax.plot(t_days, total_inventory * species_mass[sp], label=f"I_OFC+IFC+STOR_{sp}")
I_T_orig = (orig["N_ofc"] + orig["N_ifc"] + orig["N_stor"]) * species_mass["T"]
ax.plot(orig["t"] / 86400.0, I_T_orig, "k--", lw=1.8, label="I_T original ref")
ax.set_xlabel("Time (days)")
ax.set_ylabel("Total inventory (kg)")
ax.set_yscale("log")
ax.set_ylim(bottom=1e-6)
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Fusion powers
fig, ax = plt.subplots(1, 1, figsize=(9, 4.5))
t_days = np.asarray(res_dd["t"], dtype=float) / 86400.0
reaction_series = [
    ("P_DDn", P_DDn),
    ("P_DDp", P_DDp),
    ("P_DT", P_DT),
    ("P_DHe3", P_DHe3),
    ("P_TT", P_TT),
    ("P_He3He3", P_He3He3),
    ("P_THe3_ch1", P_THe3_ch1),
    ("P_THe3_ch2", P_THe3_ch2),
    ("P_THe3_ch3", P_THe3_ch3),
    ("P_fusion_total", P_fusion_total),
]
for label, series in reaction_series:
    ax.plot(t_days, np.asarray(series, dtype=float) * 1.0e-6, label=label)
ax.set_xlabel("Time (days)")
ax.set_ylabel("Power (MW)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()
